# Lesson 04 — When to Use Which: The Decision Framework

## Why This Lesson
Knowing the theory is not enough. You need a fast mental checklist for which threshold to reach for
given any input image.

## The Decision Tree
```
Is lighting uniform across the whole image?
├── YES → Is foreground/background clearly separated?
│         ├── YES → Use Otsu (automatic, optimal)
│         └── NO  → Apply CLAHE first, then Otsu
└── NO  → Use Adaptive Gaussian (always)
           └── If text on paper → blockSize=11, C=2 is almost always right
```

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def smart_threshold(img_gray):
    mean = img_gray.mean()
    std  = img_gray.std()

    # Low std = low contrast → need CLAHE first
    if std < 40:
        print("Low contrast detected → applying CLAHE before threshold")
        clahe    = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        img_gray = clahe.apply(img_gray)

    # Check if histogram is bimodal (suitable for Otsu)
    hist = cv2.calcHist([img_gray], [0], None, [256], [0,256]).flatten()
    hist_norm = hist / hist.sum()
    # Simple bimodal check: variance is high
    if std > 50:
        print("Bimodal histogram → using Otsu")
        _, result = cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        print("Uneven/low-contrast → using Adaptive Gaussian")
        result = cv2.adaptiveThreshold(img_gray, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

    return result

# Test on different image types
img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

result = smart_threshold(gray)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1); plt.imshow(gray, cmap='gray'); plt.title('Input'); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(result, cmap='gray'); plt.title('Smart Threshold Result'); plt.axis('off')
plt.show()

## Summary Table
| Situation | Method |
|---|---|
| Good contrast, uniform lighting | `THRESH_OTSU` |
| Low contrast | CLAHE → `THRESH_OTSU` |
| Uneven lighting / shadows | `ADAPTIVE_THRESH_GAUSSIAN_C` |
| Scanned text / documents | `ADAPTIVE_THRESH_GAUSSIAN_C`, blockSize=11, C=2 |
| Real-time video, no lighting control | `ADAPTIVE_THRESH_GAUSSIAN_C` |

## Key Takeaway
Adaptive Gaussian is the safe default for real-world images. Otsu is better when you have
controlled lighting and need a clean, single decision boundary.